# 01 — Preparación de Datos
**IA Generativa y Saber Pro 2021-2024** | Eduardo A. Victoria Cadena

Este notebook carga los microdatos de DataICFES, filtra la muestra (9 IES, 3 programas, 4 años) y construye todas las variables del modelo.

In [ ]:
# ── Instalar paquetes extra (solo Colab) ──────────────────────────────────
import subprocess, sys
for pkg in ["statsmodels","pyreadstat","openpyxl"]:
    subprocess.check_call([sys.executable,"-m","pip","install",pkg,"-q"])
print("Paquetes listos.")

In [ ]:
# ── Montar Google Drive ────────────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

# Ajustar a la ruta real dentro de su Drive
RUTA_PROYECTO = '/content/drive/MyDrive/IA-Y-EDUCACION-SUPERIOR'
import sys, os
sys.path.insert(0, RUTA_PROYECTO + '/python')
os.makedirs(f'{RUTA_PROYECTO}/data/processed', exist_ok=True)
os.makedirs(f'{RUTA_PROYECTO}/outputs/tablas',  exist_ok=True)
print("Drive montado en:", RUTA_PROYECTO)

In [ ]:
# ── Importaciones ─────────────────────────────────────────────────────────
import numpy  as np
import pandas as pd
import re, glob
from utils import (IES_MUESTRA, PROGRAMAS_MUESTRA, ANOS_PREVIO, ANOS_IA,
                   DISTANCIAS_BOGOTA, MODULOS_GENERICOS,
                   calcular_puntaje_generico, guardar_tabla, RUTA_PROYECTO)

RAW_DIR  = f'{RUTA_PROYECTO}/data/raw'
PROC_DIR = f'{RUTA_PROYECTO}/data/processed'

In [ ]:
# ── 1. Cargar archivos DataICFES (CSV / SAV / XLSX) ───────────────────────
def cargar_datos_brutos(ruta_raw):
    archivos = (glob.glob(f'{ruta_raw}/saberpro_*.csv')
              + glob.glob(f'{ruta_raw}/saberpro_*.sav')
              + glob.glob(f'{ruta_raw}/saberpro_*.xlsx'))
    if not archivos:
        raise FileNotFoundError(
            f"No se encontraron archivos en {ruta_raw}.\n"
            "Descargue saberpro_2021.csv ... saberpro_2024.csv de DataICFES.")
    dfs = []
    for f in archivos:
        anio = int(re.search(r'(\d{4})', os.path.basename(f)).group(1))
        ext  = f.rsplit('.',1)[-1].lower()
        if ext == 'csv':
            df = pd.read_csv(f, encoding='utf-8', low_memory=False)
        elif ext == 'sav':
            import pyreadstat
            df, _ = pyreadstat.read_sav(f)
        else:
            df = pd.read_excel(f)
        df['ANIO_APLICACION'] = anio
        dfs.append(df)
        print(f"  Cargado {os.path.basename(f)}: {len(df):,} filas")
    return pd.concat(dfs, ignore_index=True)

In [ ]:
# ── 2. Mapeo de nombres de columnas ───────────────────────────────────────
MAPA_COLS = {
    'MOD_LECTURA_CRITICA_PUNT' : 'punt_lectura_critica',
    'MOD_RAZONA_CUANTITAT_PUNT': 'punt_razona_cuant',
    'MOD_COMPETEN_CIUDADA_PUNT': 'punt_competen_ciud',
    'MOD_COMUNI_ESCRITA_PUNT'  : 'punt_comuni_escrita',
    'MOD_INGLES_PUNT'          : 'punt_ingles',
    'ESTU_CONSECUTIVO'         : 'id_estudiante',
    'ESTU_INST_NOMBRE'         : 'nombre_ies',
    'INST_ORIGEN'              : 'naturaleza_ies_raw',
    'INST_MUNICIPIO_NOMBRE'    : 'municipio_ies',
    'INST_DEPARTAMENTO_NOMBRE' : 'departamento_ies',
    'ESTU_PRGM_ACADEMICO'      : 'programa_academico',
    'FAMI_ESTRATOVIVIENDA'     : 'estrato_raw',
    'ESTU_GENERO'              : 'genero_raw',
    'FAMI_EDUCACIONPADRE'      : 'nivel_educ_padre_raw',
    'ESTU_HORASSEMANATRABAJA'  : 'horas_trabaja_raw',
    'ESTU_PAGOMATRICULAPADRES' : 'pago_matricula_raw',
    'FAMI_TIENEINTERNET'       : 'internet_raw',
    'ESTU_AREARESIDE'          : 'area_residencia_raw',
    'ESTU_PUNTAJE_GLOBAL'      : 'puntaje_saber11',
}

def mapear_columnas(df):
    mapa = {k.upper(): v for k, v in MAPA_COLS.items()}
    df.columns = [mapa.get(c.upper(), c) for c in df.columns]
    return df

In [ ]:
# ── 3. Filtrar muestra ────────────────────────────────────────────────────
def filtrar_muestra(df):
    ies_pat  = '|'.join([re.escape(i) for i in IES_MUESTRA])
    prog_pat = '|'.join([re.escape(p) for p in PROGRAMAS_MUESTRA])
    anos     = ANOS_PREVIO + ANOS_IA

    df = df.copy()
    df['_ies_up']  = df['nombre_ies'].str.upper().str.strip()
    df['_prog_up'] = df['programa_academico'].str.upper().str.strip()
    df['_dept_up'] = df['departamento_ies'].str.upper().str.strip()

    mask = (df['_ies_up'].str.contains(ies_pat, na=False)
          & df['_prog_up'].str.contains(prog_pat, na=False)
          & df['ANIO_APLICACION'].isin(anos))
    return df[mask].drop(columns=['_ies_up','_prog_up','_dept_up'])

In [ ]:
# ── 4. Construcción de variables ──────────────────────────────────────────
def construir_variables(df):
    df = df.copy()

    # periodo_ia
    df['periodo_ia'] = df['ANIO_APLICACION'].isin(ANOS_IA).astype(int)

    # estrato (numérico 1-6)
    def parse_estrato(x):
        try:
            n = int(re.search(r'\d', str(x)).group())
            return n if 1 <= n <= 6 else np.nan
        except: return np.nan
    df['estrato'] = df['estrato_raw'].apply(parse_estrato)

    # genero (0=F, 1=M)
    gmap = {'M':1,'MASCULINO':1,'HOMBRE':1,'F':0,'FEMENINO':0,'MUJER':0}
    df['genero'] = df['genero_raw'].str.upper().str.strip().map(gmap)

    # nivel_educ_padre (ordinal 1-7)
    def parse_educ(x):
        x = str(x).upper()
        if any(k in x for k in ['NINGUNO','SIN EDUC']): return 1
        if 'PRIMARIA COMP' in x: return 2
        if 'PRIMARIA' in x: return 1
        if any(k in x for k in ['SECUND COMP','BACHILLERATO COMP']): return 4
        if any(k in x for k in ['SECUND','BACHILLERATO']): return 3
        if any(k in x for k in ['TECNIC','TECNOLOG']): return 5
        if any(k in x for k in ['UNIVERSIT','PROFESION']): return 6
        if any(k in x for k in ['POSTGRADO','ESPECIALI','MAESTR','DOCTOR']): return 7
        return np.nan
    df['nivel_educ_padre'] = df['nivel_educ_padre_raw'].apply(parse_educ)

    # estu_trabaja
    def parse_trabaja(x):
        x = str(x).upper()
        if '0' in x or 'NO TRABAJ' in x: return 0
        if pd.isna(x) or x == 'NAN': return np.nan
        return 1
    df['estu_trabaja'] = df['horas_trabaja_raw'].apply(parse_trabaja)

    # estu_cabeza_familia (proxy: paga su propia matrícula)
    def parse_cabeza(x):
        x = str(x).upper()
        if any(k in x for k in ['EL MISMO','PROPIO','ESTUDIANTE']): return 1
        if x == 'NAN': return np.nan
        return 0
    df['estu_cabeza_familia'] = df['pago_matricula_raw'].apply(parse_cabeza)

    # internet
    imap = {'SI':1,'SÍ':1,'S':1,'NO':0,'N':0}
    df['internet'] = df['internet_raw'].str.upper().str.strip().map(imap)

    # area_residencia
    amap = {'URBANA':1,'CABECERA':1,'RURAL':0,'CENTRO POBLADO':0,'RURAL DISPERSO':0}
    df['area_residencia'] = df['area_residencia_raw'].str.upper().str.strip().map(amap)

    # naturaleza_ies
    nmap = {'OFICIAL':0,'PUBLICA':0,'PRIVADA':1}
    df['naturaleza_ies'] = df['naturaleza_ies_raw'].str.upper().str.strip().map(nmap)

    # departamento
    def parse_depto(x):
        x = str(x).upper()
        if 'BOGOT'    in x: return 'BOGOTA'
        if 'ANTIOQUI' in x: return 'ANTIOQUIA'
        if 'VALLE'    in x: return 'VALLE'
        if 'HUILA'    in x: return 'HUILA'
        if 'NARI'     in x: return 'NARINO'
        if 'TOLIMA'   in x: return 'TOLIMA'
        return 'OTRO'
    df['depto_ies'] = df['departamento_ies'].apply(parse_depto)

    # Dummies departamentales (Bogotá = referencia)
    for d in ['antioquia','valle','huila','narino','tolima']:
        df[f'd_{d}'] = (df['depto_ies'] == d.upper()).astype(int)

    # distancia_bogota_km
    df['distancia_bogota_km'] = df['depto_ies'].map(DISTANCIAS_BOGOTA)

    # puntaje_saber11: asegurar numérico
    df['puntaje_saber11'] = pd.to_numeric(df['puntaje_saber11'], errors='coerce')

    # Puntajes módulos: numérico
    for m in MODULOS_GENERICOS:
        if m in df.columns:
            df[m] = pd.to_numeric(df[m], errors='coerce')

    # Puntaje genérico agregado
    df = calcular_puntaje_generico(df)
    return df

In [ ]:
# ── 5. Depuración final ───────────────────────────────────────────────────
def depurar_datos(df):
    antes = len(df)
    df = df[df['puntaje_saberpro_generico'].notna()].copy()
    print(f"Eliminados por NA en puntaje: {antes - len(df)}")

    # Rango válido 0-300
    for m in MODULOS_GENERICOS:
        if m in df.columns:
            df = df[(df[m].between(0,300)) | df[m].isna()]

    print(f"Observaciones finales: {len(df):,}")
    print("\nDistribución por año:")
    print(df['ANIO_APLICACION'].value_counts().sort_index())
    print("\nDistribución por departamento:")
    print(df['depto_ies'].value_counts())
    return df

In [ ]:
# ── 6. Datos simulados (prueba sin DataICFES real) ────────────────────────
def generar_datos_simulados(n=3000, semilla=42):
    rng = np.random.default_rng(semilla)
    deptos = list(DISTANCIAS_BOGOTA.keys())
    prob_d = [0.30,0.20,0.20,0.12,0.10,0.08]
    ies_map = {
        'BOGOTA'   :['UNIVERSIDAD NACIONAL DE COLOMBIA','UNIVERSIDAD DISTRITAL FRANCISCO JOSE DE CALDAS'],
        'ANTIOQUIA':['UNIVERSIDAD DE ANTIOQUIA'],
        'VALLE'    :['UNIVERSIDAD DEL VALLE'],
        'HUILA'    :['UNIVERSIDAD SURCOLOMBIANA'],
        'NARINO'   :['UNIVERSIDAD DE NARINO'],
        'TOLIMA'   :['UNIVERSIDAD DEL TOLIMA'],
    }
    depto_arr = rng.choice(deptos, n, p=prob_d)
    anio_arr  = rng.choice([2021,2022,2023,2024], n)
    df = pd.DataFrame({
        'id_estudiante'  : range(n),
        'ANIO_APLICACION': anio_arr,
        'depto_ies'      : depto_arr,
        'nombre_ies'     : [rng.choice(ies_map[d]) for d in depto_arr],
        'programa_academico': rng.choice(['ECONOMIA','ADMINISTRACION DE EMPRESAS',
                                          'CONTADURIA PUBLICA'], n, p=[0.25,0.45,0.30]),
        'periodo_ia'     : (pd.Series(anio_arr).isin(ANOS_IA)).astype(int).values,
        'estrato'        : rng.choice(range(1,7), n, p=[0.10,0.25,0.30,0.20,0.10,0.05]),
        'genero'         : rng.binomial(1, 0.48, n),
        'nivel_educ_padre': rng.choice(range(1,8), n,
                              p=[0.05,0.10,0.15,0.30,0.18,0.15,0.07]),
        'estu_trabaja'   : rng.binomial(1, 0.40, n),
        'estu_cabeza_familia': rng.binomial(1, 0.15, n),
        'internet'       : rng.binomial(1, 0.82, n),
        'area_residencia': rng.binomial(1, 0.78, n),
        'naturaleza_ies' : np.zeros(n, dtype=int),
        'puntaje_saber11': rng.normal(280, 45, n),
    })
    df['distancia_bogota_km'] = df['depto_ies'].map(DISTANCIAS_BOGOTA)
    for d in ['antioquia','valle','huila','narino','tolima']:
        df[f'd_{d}'] = (df['depto_ies'] == d.upper()).astype(int)

    efecto_ia    = -2.5 * df['periodo_ia']
    efecto_depto = df['depto_ies'].map({'BOGOTA':0,'ANTIOQUIA':-4,'VALLE':-5,
                                         'HUILA':-9,'NARINO':-14,'TOLIMA':-7})
    base = 145 + efecto_ia + 3*df['estrato'] + efecto_depto + 0.05*(df['puntaje_saber11']-280)
    for i, m in enumerate(MODULOS_GENERICOS):
        offset = [0,-5,2,1,-8][i]
        df[m] = np.clip(base + offset + rng.normal(0,12,n), 0, 300)
    df = calcular_puntaje_generico(df)
    return df

In [ ]:
# ── 7. Flujo principal ────────────────────────────────────────────────────
import os

archivos_raw = (glob.glob(f'{RAW_DIR}/saberpro_*.csv')
              + glob.glob(f'{RAW_DIR}/saberpro_*.sav')
              + glob.glob(f'{RAW_DIR}/saberpro_*.xlsx'))

if archivos_raw:
    print("Datos reales detectados. Procesando...")
    datos_brutos   = cargar_datos_brutos(RAW_DIR)
    datos_mapeados = mapear_columnas(datos_brutos)
    datos_filtrados= filtrar_muestra(datos_mapeados)
    datos_construidos = construir_variables(datos_filtrados)
    df = depurar_datos(datos_construidos)
else:
    print("⚠ No se encontraron datos en data/raw/")
    print("Usando datos SIMULADOS (solo para probar el flujo).")
    df = generar_datos_simulados(n=3000)

df.to_csv(f'{PROC_DIR}/datos_analisis.csv', index=False)
df.to_pickle(f'{PROC_DIR}/datos_analisis.pkl')
print(f"\nMuestra final: {len(df):,} filas | {df.shape[1]} columnas")
print(f"pre-IA (periodo_ia=0): n={( df['periodo_ia']==0).sum():,}")
print(f"IA     (periodo_ia=1): n={(df['periodo_ia']==1).sum():,}")
df.head()